## Vectorisation des paragraphes 

Format avec modeles Em de base et fine tuné 

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [11]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

import json


In [2]:
# ── Chemins ───────────────────────────────────────────────────────────────────
CSV_PATH        = r"DATA\df_paragraphe_final.csv"          # chemin vers ton fichier CSV         # chemin vers les poids du modèle fine-tuné
OUTPUT_BASE     = "embeddings_base.npz"       # sortie vectorisation modèle de base
OUTPUT_FINETUNED= "embeddings_finetuned.npz"  # sortie vectorisation modèle fine-tuné

# ── Paramètres ────────────────────────────────────────────────────────────────
MODEL_NAME  = "intfloat/multilingual-e5-large"
BATCH_SIZE  = 32
PREFIX      = "passage: "   # préfixe attendu par e5 pour les passages à indexer
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")

Device : cuda


In [9]:
def load_corpus(csv_path: str) -> pd.DataFrame:
    """Charge le CSV et affiche un résumé."""
    df = pd.read_csv(csv_path)
    assert {"paragraphe", "nom_du_fichier", "page"}.issubset(df.columns), \
        "Colonnes attendues : paragraphe, nom_du_fichier, page"
    df = df.dropna(subset=["paragraphe"]).reset_index(drop=True)
    print(f"Corpus chargé : {len(df)} paragraphes | {df['nom_du_fichier'].nunique()} fichiers")
    return df


def add_prefix(texts: list[str], prefix: str) -> list[str]:
    """Ajoute le préfixe e5 à chaque texte."""
    return [prefix + t for t in texts]


def vectorize(
    model: SentenceTransformer,
    texts: list[str],
    batch_size: int = 32,
    prefix: str = "passage: ",
    device: str = "cpu",
) -> np.ndarray:
    """
    Vectorise une liste de textes par batchs.
    Retourne un array numpy (n_textes, dim_embedding), normalisé L2.
    """
    model.to(device)
    model.eval()

    prefixed = add_prefix(texts, prefix)
    all_embs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(prefixed), batch_size), desc="Vectorisation"):
            batch = prefixed[i : i + batch_size]
            embs  = model.encode(
                batch, 
                batch_size=batch_size,
                normalize_embeddings=True,
                convert_to_numpy=True,
                device=device,
                show_progress_bar=False,
            )
            all_embs.append(embs)

    return np.vstack(all_embs)


def save_embeddings(embeddings: np.ndarray, df: pd.DataFrame, output_path: str):
    """Sauvegarde embeddings + métadonnées dans un fichier .npz"""
    np.savez(
        output_path,
        embeddings     = embeddings,
        paragraphes    = df["paragraphe"].values,
        noms_fichiers  = df["nom_du_fichier"].values,
        pages          = df["page"].values,
    )
    print(f"Sauvegardé → {output_path}  |  shape : {embeddings.shape}")


df = load_corpus(CSV_PATH)

Corpus chargé : 77777 paragraphes | 372 fichiers


#### Modele de base

In [ ]:


model_base = SentenceTransformer(MODEL_NAME)

embeddings_base = vectorize(
    model     = model_base,
    texts     = df["paragraphe"].tolist(),
    batch_size= BATCH_SIZE,
    prefix    = PREFIX,
    device    = DEVICE,
)

save_embeddings(embeddings_base, df, OUTPUT_BASE)

#### Modele finetuné

In [10]:
from sentence_transformers import SentenceTransformer

model_finetuned = SentenceTransformer("cocongy/e5-finetuned") 

embeddings_finetuned = vectorize(
    model     = model_finetuned,
    texts     = df["paragraphe"].tolist(),
    batch_size= BATCH_SIZE,
    prefix    = PREFIX,
    device    = DEVICE,
)

save_embeddings(embeddings_finetuned, df, OUTPUT_FINETUNED)

This model was created with Sentence Transformers version 5.5.1, but you're using version 5.5.0. Consider updating to the latest version to avoid potential issues.
Vectorisation: 100%|██████████| 2431/2431 [21:04<00:00,  1.92it/s]  


Sauvegardé → embeddings_finetuned.npz  |  shape : (77777, 1024)


# Embbeding question 

In [18]:
# ── Chemins ───────────────────────────────────────────────────────────────────
QUESTIONS_PATH       = r"donné_pour_test\20260430_task1_test_query.json"
CSV_PATH             = r"DATA\df_paragraphe_final.csv"
EMB_BASE_PATH        = r"embeddings_base.npz"
EMB_FINETUNED_PATH   = r"embeddings_finetuned.npz"

OUTPUT_BASE          = r"resultat\resultats_base.csv"
OUTPUT_FINETUNED     = r"resultat\resultats_finetuned.csv"

# ── Paramètres ────────────────────────────────────────────────────────────────
MODEL_BASE_NAME = "intfloat/multilingual-e5-large"
K               = 5
PREFIX_QUERY    = "query: "   # préfixe e5 pour les questions
BATCH_SIZE      = 32
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")

# ── Chargement questions ───────────────────────────────────────────────────────
with open(QUESTIONS_PATH, "r", encoding="utf-8") as f:
    questions_data = json.load(f)

questions_data = questions_data["results"]
questions = [item["question"] for item in questions_data]
qids      = [item["qid"]      for item in questions_data]

print(f"{len(questions)} questions chargées")

# ── Chargement embeddings paragraphes ─────────────────────────────────────────
def load_npz(path: str):
    data = np.load(path, allow_pickle=True)
    embeddings   = data["embeddings"]          # (N, dim)
    paragraphes  = data["paragraphes"]         # (N,)
    noms_fichiers= data["noms_fichiers"]       # (N,)
    pages        = data["pages"]               # (N,)
    print(f"Embeddings chargés depuis {path} | shape : {embeddings.shape}")
    return embeddings, paragraphes, noms_fichiers, pages

emb_base,      para_base,      noms_base,      pages_base      = load_npz(EMB_BASE_PATH)
emb_finetuned, para_finetuned, noms_finetuned, pages_finetuned = load_npz(EMB_FINETUNED_PATH)

Device : cuda
595 questions chargées
Embeddings chargés depuis embeddings_base.npz | shape : (77777, 1024)
Embeddings chargés depuis embeddings_finetuned.npz | shape : (77777, 1024)


In [19]:
def embed_questions(
    model: SentenceTransformer,
    questions: list[str],
    prefix: str  = "query: ",
    batch_size: int = 32,
    device: str = "cpu",
) -> np.ndarray:
    """
    Encode les questions avec le préfixe e5 adapté aux requêtes.
    Retourne un array (n_questions, dim), normalisé L2.
    """
    model.to(device)
    model.eval()

    prefixed = [prefix + q for q in questions]
    all_embs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(prefixed), batch_size), desc="Embedding questions"):
            batch = prefixed[i : i + batch_size]
            embs  = model.encode(
                batch,
                batch_size=batch_size,
                normalize_embeddings=True,
                convert_to_numpy=True,
                device=device,
                show_progress_bar=False,
            )
            all_embs.append(embs)

    return np.vstack(all_embs)


def top_k_similarities(
    q_embeddings: np.ndarray,      # (n_questions, dim)
    p_embeddings: np.ndarray,      # (N_paragraphes, dim)
    paragraphes:  np.ndarray,
    noms_fichiers:np.ndarray,
    pages:        np.ndarray,
    qids:         list[str],
    questions:    list[str],
    k: int = 5,
) -> pd.DataFrame:
    """
    Calcule le cosine similarity (dot product sur vecteurs L2-normalisés)
    entre chaque question et tous les paragraphes, puis retourne les K meilleurs.
    """
    # scores : (n_questions, N_paragraphes)
    scores = q_embeddings @ p_embeddings.T

    rows = []
    for i, (qid, question) in enumerate(zip(qids, questions)):
        top_k_idx    = np.argsort(scores[i])[::-1][:k]
        top_paras    = [paragraphes[j]   for j in top_k_idx]
        top_noms     = [noms_fichiers[j] for j in top_k_idx]
        top_scores   = [round(float(scores[i][j]), 4) for j in top_k_idx]

        row = {"qid": qid, "question": question}
        for rank, (para, nom, score) in enumerate(zip(top_paras, top_noms, top_scores), start=1):
            row[f"paragraphe_{rank}"]    = para
            row[f"nom_document_{rank}"]  = nom
            row[f"score_{rank}"]         = score

        rows.append(row)

    return pd.DataFrame(rows)


def save_results(df: pd.DataFrame, output_path: str):
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Résultats sauvegardés → {output_path}  |  {len(df)} lignes")

In [20]:
# ── Embedding des questions (modèle de base) ──────────────────────────────────
model_base       = SentenceTransformer(MODEL_BASE_NAME)
q_emb_base       = embed_questions(model_base, questions, prefix=PREFIX_QUERY,
                                   batch_size=BATCH_SIZE, device=DEVICE)
np.save("resultat/q_embeddings_base.npy", q_emb_base)

# ── Top-K similarités ─────────────────────────────────────────────────────────
df_results_base  = top_k_similarities(
    q_embeddings  = q_emb_base,
    p_embeddings  = emb_base,
    paragraphes   = para_base,
    noms_fichiers = noms_base,
    pages         = pages_base,
    qids          = qids,
    questions     = questions,
    k             = K,
)

# ── Rendu du tableau ──────────────────────────────────────────────────────────
display(df_results_base.head(3))
save_results(df_results_base, OUTPUT_BASE)

Embedding questions: 100%|██████████| 19/19 [00:05<00:00,  3.80it/s]


,qid,question,paragraphe_1,nom_document_1,score_1,paragraphe_2,nom_document_2,score_2,paragraphe_3,nom_document_3,score_3,paragraphe_4,nom_document_4,score_4,paragraphe_5,nom_document_5,score_5
0,Q1,Quelles sont les trois catégories principales ...,"stallées sur des aéronefs, ayant pour seule fi...",Nouvelle réglementation relative à la protecti...,0.8841,"rrains militaires, elle relève de la protectio...",Manuel de droit des opérations militaires.pdf,0.8809,"érêts fondamentaux, soit un détournement à des...",Manuel de droit des opérations militaires.pdf,0.8741,n au sein d’une installation militaire située ...,Manuel de droit des opérations militaires.pdf,0.8723,e spécifique des ZDHS leur apporte un régime d...,Manuel de droit des opérations militaires.pdf,0.8717
1,Q2,Quelles sont les trois catégories de satelites...,ses pourrait constituer un champ d’opérations ...,Drones navales DENIS.pdf,0.8783,illance and Tracking : 277. Status of Force Ag...,Manuel de droit des opérations militaires.pdf,0.8717,"duaux de l’adversaire, mais aussi des dommages...",Manuel de droit des opérations militaires.pdf,0.8687,M voire une standardisation de cette entrepris...,"l'EPS 2021-08 M2MC enjeux, opportunités et ris...",0.8685,"morts, des blessures et/ou des destructions pa...",Manuel de droit des opérations militaires.pdf,0.8649
2,Q3,stratégies désinformation faux médias noms cré...,.com/daniel_gugger x.com/vtforeignpolicy t.me/...,20250507_TLP-CLEAR_NP_SGDSN_VIGINUM_Rapport te...,0.8796,"nipulation de l’information », News: Stakehold...",LMI_PEC_2023.pdf,0.8761,[T0093.001] Fund Proxies - [T0118] Amplify Exi...,20250507_TLP-CLEAR_NP_SGDSN_VIGINUM_Rapport te...,0.8758,"taux, des exigences de proportionnalité et des...",Stratégie nationale de lutte contre les manipu...,0.8742,> MANIPULATION DE L'INFORMATION: ensemble des ...,CDSE_VIGINUM_Guide_sensibilisation_entreprises...,0.8741


Résultats sauvegardés → resultat\resultats_base.csv  |  595 lignes


In [21]:
# ── Embedding des questions (modèle fine-tuné) ────────────────────────────────
model_finetuned  = SentenceTransformer("cocongy/e5-finetuned")
q_emb_finetuned  = embed_questions(model_finetuned, questions, prefix=PREFIX_QUERY,
                                   batch_size=BATCH_SIZE, device=DEVICE)
np.save("resultat/q_embeddings_finetuned.npy", q_emb_finetuned)

# ── Top-K similarités ─────────────────────────────────────────────────────────
df_results_finetuned = top_k_similarities(
    q_embeddings  = q_emb_finetuned,
    p_embeddings  = emb_finetuned,
    paragraphes   = para_finetuned,
    noms_fichiers = noms_finetuned,
    pages         = pages_finetuned,
    qids          = qids,
    questions     = questions,
    k             = K,
)

# ── Rendu du tableau ──────────────────────────────────────────────────────────
display(df_results_finetuned.head(3))
save_results(df_results_finetuned, OUTPUT_FINETUNED)

This model was created with Sentence Transformers version 5.5.1, but you're using version 5.5.0. Consider updating to the latest version to avoid potential issues.
Embedding questions: 100%|██████████| 19/19 [00:04<00:00,  3.85it/s]


,qid,question,paragraphe_1,nom_document_1,score_1,paragraphe_2,nom_document_2,score_2,paragraphe_3,nom_document_3,score_3,paragraphe_4,nom_document_4,score_4,paragraphe_5,nom_document_5,score_5
0,Q1,Quelles sont les trois catégories principales ...,"s équipements, moyens de transport ou dépôts c...",Manuel de droit des opérations militaires.pdf,0.9006,n au sein d’une installation militaire située ...,Manuel de droit des opérations militaires.pdf,0.8913,"bâtiments d’enseignement ou de sport, etc. > L...",RNCE_19_Juillet_2023.pdf,0.8858,n déterminé par le ministre compétent. La zone...,2011113_IGI 1300_Protection du secret de la de...,0.8855,n’être accessible qu'aux personnes autorisées ...,2011113_IGI 1300_Protection du secret de la de...,0.8853
1,Q2,Quelles sont les trois catégories de satelites...,"ns de renseignement électronique Sigint, ainsi...",onera4.pdf,0.8576,"de la structure), de performances (matériaux),...",KAMMOUN_Concept d'emploi des drones au sein de...,0.8563,"objets spatiaux, pouvant déclencher une réacti...",Manuel de droit des opérations militaires.pdf,0.8511,"erme, en cas de dommages au sol suite à la chu...",onera4.pdf,0.8448,"(formes de la structure), de performances (mat...",onera4.pdf,0.8445
2,Q3,stratégies désinformation faux médias noms cré...,TLP:CLEAR Sommaire 1. Introduction ..............,20260122_NP_TLP-CLEAR_SGDSN_VIGINUM_MOI.pdf,0.9083,"onstituées et coordonnées par un gouvernement,...",LMI_PEC_2023.pdf,0.9064,la majeure partie est apparue en corollaire de...,20250224_TLP-CLEAR_NP_SGDSN_VIGINUM_Guerre en ...,0.9054,ternational structuré autour du concept de FIM...,Stratégie nationale de lutte contre les manipu...,0.9015,"te contre la manipulation de l’information, se...",Stratégie nationale de lutte contre les manipu...,0.8984


Résultats sauvegardés → resultat\resultats_finetuned.csv  |  595 lignes
